# einops-rearrange — ex10: rearrange single-token K/V into KV-cache layout for decode-time concat

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. Running the final beacon cell reports progress against the `Einops: Rearrange` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## rearrange for KV-cache concat — quick refresher

At LLM decode time, each new token's K and V projections are concatenated onto a growing **KV cache**. The cache layout is `(B, H, S_total, D)`; the new token's projection arrives as `(B, H, D)` (a single timestep). The standard pattern:

```python
# new_k: (B, H, D) → (B, H, 1, D), then cat onto cache along the S axis
new_k_s = rearrange(new_k, 'b h d -> b h 1 d')
cache_k = t.cat([cache_k, new_k_s], dim=2)
```

**Compared to `unsqueeze(2)`.** Both produce the same shape. `rearrange` makes the intent explicit at the call site — readers see `b h d -> b h 1 d` and immediately know 'we're materializing a singleton sequence axis'. `unsqueeze(2)` requires you to remember what dim 2 means.

**This drill (ex10) vs ex1-9.** Earlier exercises did ViT-style patchify, NHWC↔NCHW, divisibility edge cases. ex10 covers the DECODE-LOOP rearrange — packing single-token K/V into the right cache axis on every generation step.

### Exercise 10 — rearrange single-token K/V into KV-cache layout for decode-time concat

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `rearrange(... 'b h d -> b h 1 d')` to materialize a singleton sequence axis on each new token's K projection, then concat onto a growing KV cache across decode steps.
> Keywords: kv-cache, decode-loop, singleton-axis, rearrange
> ```

**KCs targeted:** `rearrange-singleton-axis-literal`, `rearrange-decode-time-pattern`

Implement `ex10_kv_cache_decode(initial_cache_k, new_tokens_k)`.

Simulate `n_steps` decode iterations of a transformer's KV-cache concat for the K tensor:

1. `initial_cache_k` has shape `(B, H, S0, D)` — the cache BEFORE decoding starts (may be 0-length on first call if `S0 == 0`).
2. `new_tokens_k` has shape `(n_steps, B, H, D)` — one new K projection per decode step.
3. For each step `i` in `range(n_steps)`:
   a. Extract `new_k = new_tokens_k[i]` (shape `(B, H, D)`).
   b. Use `rearrange(new_k, 'b h d -> b h 1 d')` to materialize the singleton sequence axis.
   c. Concat onto `cache_k` along `dim=2`: `cache_k = t.cat([cache_k, new_k_s], dim=2)`.
4. Return `cache_k` after all `n_steps` concatenations.

Final shape: `(B, H, S0 + n_steps, D)`.

The visualization plots `cache_k.shape[2]` (the sequence axis length) over each step to show the cache growing linearly.

In [ ]:
def ex10_kv_cache_decode(
    initial_cache_k: Tensor,
    new_tokens_k: Tensor,
) -> Tensor:
    cache_k = initial_cache_k
    for i in range(new_tokens_k.shape[0]):
        new_k = new_tokens_k[i]                                  # (B, H, D)
        new_k_s = rearrange(new_k, 'b h d -> b h 1 d')           # (B, H, 1, D)
        cache_k = t.cat([cache_k, new_k_s], dim=2)
    return cache_k


<details><summary>Solution</summary>

```python
def ex10_kv_cache_decode(
    initial_cache_k: Tensor,
    new_tokens_k: Tensor,
) -> Tensor:
    cache_k = initial_cache_k
    for i in range(new_tokens_k.shape[0]):
        new_k = new_tokens_k[i]                                  # (B, H, D)
        new_k_s = rearrange(new_k, 'b h d -> b h 1 d')           # (B, H, 1, D)
        cache_k = t.cat([cache_k, new_k_s], dim=2)
    return cache_k
```

**Why a singleton axis instead of just `t.cat([..., dim=...])`.** `cat` requires every input to have the same number of dims. The fresh K projection has rank 3 `(B, H, D)`; the cache has rank 4. Without rearrange you'd unsqueeze; with rearrange the intent ('I am creating a length-1 sequence axis') is documented at the call site.

**Why iterate rather than vectorize.** Decode is inherently sequential — token `t+1` depends on the model's attention to all tokens through `t`. You can't batch decode across timesteps without speculative-decoding tricks. The cache concat IS the inner loop of every transformer inference engine.

**Memory cost.** `t.cat` allocates a new tensor every step. Production engines (vLLM, TensorRT-LLM) pre-allocate a max-length cache and overwrite slots in place to avoid the O(N) allocation overhead. The rearrange-then-cat shape here is the *correctness reference*; the prod fast-path is the same pattern with the cat replaced by an indexed assignment into a pre-sized buffer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()